# MODIS Batch-Style Playground

Use this notebook as a compact batch-ready example for MODIS Terra/Aqua processing with SpiPy.

Workflow:
- configure `sensor`, `tile`, `year`, and the shared MODIS LUT
- build or reuse a saved `R0` under `data/modis/r0/<sensor>/<tile>/<year>/`
- loop over matching inversion scenes under `data/modis/inputs/<sensor>/reflectance/`
- compute each inversion, save a 1x4 quicklook figure, and write a netCDF result
- inspect the saved outputs and recent workflow log lines


In [ ]:
from pathlib import Path
import sys
from time import perf_counter

from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import xarray as xr

search_roots = [Path.cwd(), *Path.cwd().parents]
repo_root = next(path for path in search_roots if (path / 'spires').exists() and (path / 'tests' / 'data').exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from spires import configure_spires_file_logger, make_spires_log_path
from spires.sensors.modis import (
    build_modis_r0_from_sources,
    parse_modis_surface_reflectance_filename,
    prepare_modis_scene_for_inversion,
    run_modis_inversion,
)

SENSOR_LABELS = {
    'terra': 'Terra',
    'aqua': 'Aqua',
}

sensor = 'terra'  # change to 'aqua' for MYD09GA
tile = 'h08v05'
year = 2023
lut_path = repo_root / 'tests' / 'data' / 'lut_modis_1_2_3_4_5_6_7_3um_dust_bandpass.mat'

data_root = repo_root / 'data' / 'modis'
r0_input_dir = data_root / 'inputs' / sensor / 'r0'
reflectance_input_dir = data_root / 'inputs' / sensor / 'reflectance'
r0_input_glob = '**/*.hdf'
inversion_glob = '*.hdf'

output_root = repo_root / 'outputs' / 'modis' / sensor / tile / str(year)
figure_dir = output_root / 'figures'
netcdf_dir = output_root / 'netcdf'
figure_dir.mkdir(parents=True, exist_ok=True)
netcdf_dir.mkdir(parents=True, exist_ok=True)

r0_output_dir = data_root / 'r0' / sensor / tile / str(year)
r0_output_dir.mkdir(parents=True, exist_ok=True)
r0_path = r0_output_dir / f'modis_r0_{sensor}_{tile}_{year}.nc'

r0_source_paths = sorted(r0_input_dir.glob(r0_input_glob))
scenes_for_inversion = sorted(reflectance_input_dir.glob(inversion_glob))

log_path = make_spires_log_path(output_root / 'logs', prefix=f'modis_{sensor}_{tile}_{year}')
logger = configure_spires_file_logger(
    log_path,
    logger_name=f'spires.modis.{sensor}.{tile}.{year}',
    log_to_stdout=False,
)

run_summary_lines = [
    f'Repo root: {repo_root}',
    f'Data root: {data_root}',
    f'R0 input dir: {r0_input_dir}',
    f'R0 input glob: {r0_input_glob}',
    f'R0 source scenes found: {len(r0_source_paths)}',
    f'Reflectance input dir: {reflectance_input_dir}',
    f'Inversion glob: {inversion_glob}',
    f'Scenes for inversion found: {len(scenes_for_inversion)}',
    f'Sensor: {sensor}',
    f'Tile: {tile}',
    f'Year: {year}',
    f'LUT: {lut_path}',
    f'R0 output: {r0_path}',
    f'Figure output dir: {figure_dir}',
    f'NetCDF output dir: {netcdf_dir}',
    f'Log file: {log_path}',
]

for line in run_summary_lines:
    logger.info(line)
    print(line)


## Input Check

This notebook assumes that `r0_source_paths` and `scenes_for_inversion` are non-empty and that the scenes match the configured LUT, tile, and platform. If the next cell fails, adjust `sensor`, `tile`, the directory contents, or the glob patterns first.


In [ ]:
assert r0_source_paths, f'No R0 source scenes matched {r0_input_glob!r} in {r0_input_dir}'
assert scenes_for_inversion, f'No inversion scenes matched {inversion_glob!r} in {reflectance_input_dir}'
assert lut_path.exists(), f'Missing MODIS LUT: {lut_path}'

summer_meta = [parse_modis_surface_reflectance_filename(path) for path in r0_source_paths]
inversion_meta = [parse_modis_surface_reflectance_filename(path) for path in scenes_for_inversion]
all_meta = summer_meta + inversion_meta
assert {meta.platform for meta in all_meta} == {sensor}, 'Summer and inversion scenes must use one MODIS platform.'
assert {meta.tile for meta in all_meta} == {tile}, 'Summer and inversion scenes must use one MODIS tile.'
assert {meta.collection for meta in all_meta} == {'061'}, 'This notebook expects Collection 061 MOD09GA/MYD09GA files.'

print('First three R0 source scenes:')
for path in r0_source_paths[:3]:
    print(' ', path.name)
print()
print('First three inversion scenes:')
for path in scenes_for_inversion[:3]:
    print(' ', path.name)


## Batch-Style Processing

This cell builds or reuses the `R0` file, then processes every scene matched by `inversion_glob`. For each scene it prepares the inputs, creates and computes the lazy inversion dataset, saves a 1x4 quicklook figure, and writes a sensor/date-specific netCDF.


In [ ]:
def plot_modis_inversion_1x4(inversion_ds, *, sensor, acquisition_date, figure_path):
    sensor_label = SENSOR_LABELS.get(sensor, sensor.title())
    fig, ax = plt.subplots(1, 4, figsize=(24, 5))
    (inversion_ds.raw_viewable_snow_fraction * 100).plot(ax=ax[0], cmap='YlGnBu_r', vmin=0, vmax=100)
    (inversion_ds.raw_shade_fraction * 100).plot(ax=ax[1], cmap='Greys', vmin=0, vmax=100)
    (inversion_ds.raw_canopy_fraction * 100).plot(ax=ax[2], cmap='YlGn', vmin=0, vmax=100)
    (inversion_ds.raw_snow_fraction * 100).plot(ax=ax[3], cmap='YlGnBu_r', vmin=0, vmax=100)

    ax[0].set_title('SpiPy raw viewable snow fraction')
    ax[1].set_title('SpiPy raw shade fraction')
    ax[2].set_title('SpiPy raw canopy fraction')
    ax[3].set_title('SpiPy raw snow fraction')
    fig.suptitle(f'MODIS {sensor_label} // {acquisition_date}', y=1.02)
    fig.tight_layout()
    fig.savefig(figure_path, dpi=300, bbox_inches='tight')
    plt.show()
    return fig


processed_outputs = []
r0_build_start = perf_counter()
r0_ds = build_modis_r0_from_sources(
    r0_source_paths,
    r0_path=r0_path,
    lut_file=lut_path,
    logger=logger,
    show_progress=True,
)
print(f'R0 ready in {perf_counter() - r0_build_start:.2f} s -> {r0_path}')

for scene_path in tqdm(scenes_for_inversion, desc='Running MODIS inversions'):
    meta = parse_modis_surface_reflectance_filename(scene_path)
    acquisition_date = meta.acquisition_date
    scene_tag = f'{meta.product}_{sensor}_{tile}_{acquisition_date}'
    figure_path = figure_dir / f'{scene_tag}.png'
    netcdf_path = netcdf_dir / f'{scene_tag}.nc'

    print(f'Processing {scene_path.name} ...')
    scene_start = perf_counter()
    prepared_scene = prepare_modis_scene_for_inversion(scene_path, lut_file=lut_path, logger=logger)
    lazy_inversion = run_modis_inversion(
        prepared_scene,
        r0_ds,
        lut_file=lut_path,
        execution_profile='local',
        logger=logger,
        apply_valid_inversion_mask=False,
    )
    with ProgressBar(dt=5.0):
        inversion_ds = lazy_inversion.compute()

    inversion_ds.to_netcdf(netcdf_path)
    plot_modis_inversion_1x4(
        inversion_ds,
        sensor=sensor,
        acquisition_date=acquisition_date,
        figure_path=figure_path,
    )

    processed_outputs.append(
        {
            'scene': scene_path,
            'netcdf': netcdf_path,
            'figure': figure_path,
            'elapsed_seconds': perf_counter() - scene_start,
        }
    )
    print(f'  saved netCDF -> {netcdf_path.name}')
    print(f'  saved figure -> {figure_path.name}')
    print(f'  elapsed      -> {processed_outputs[-1]["elapsed_seconds"]:.2f} s')


## Saved Outputs and Log Tail

This confirms the saved `R0` location, batch outputs, and recent workflow log events.


In [ ]:
saved_r0 = xr.open_dataset(r0_path)

print(saved_r0)
print()
print('Processed outputs:')
for item in processed_outputs:
    print(' ', item['scene'].name)
    print('    netCDF:', item['netcdf'])
    print('    figure:', item['figure'])
print()
print('Recent log lines:')
for line in log_path.read_text().splitlines()[-14:]:
    print(line)
